<a href="https://colab.research.google.com/github/CristovaoDias/LEFA-MECREL/blob/main/Notebooks_Colab/TP03_Cinematica_Curvilinea_e_Animacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Mecânica e Relatividade (LEFA-MECREL) — Aula TP 3
## 🏎️ Cinemática Curvilínea, Triedro Móvel de Frenet-Serret e Simulação Interativa com GenAI

**Licenciatura em Engenharia Física Aplicada (LEFA) — ISEP / DFI**  
**Docente:** Cristóvão de Sousa Dias (`css@isep.ipp.pt`)  
**Semestre:** 1.º Ano / 1.º Semestre (2026/2027)  

---

### 📌 Objetivos Pedagógicos desta Atividade Computacional:
1. **Visualizar dinamicamente o Triedro Móvel de Frenet-Serret** $(\hat{u}_t, \hat{u}_n)$ ao longo de trajetórias curvilíneas planas.
2. **Compreender a decomposição física do vetor aceleração** nas componentes tangencial ($a_t = \dot{v}$, taxa de variação da rapidez) e normal ($a_n = v^2/\rho$, taxa de mudança de direção).
3. **Construir interativamente o círculo de curvatura** e o raio de curvatura local $R(t)$, observando onde a trajetória é mais apertada.
4. **Praticar Engenharia de Prompts com IA Generativa** para gerar código científico de modelação de pistas e veículos de competição.
5. **Auditar fisicamente código gerado por IA**, identificando singularidades numéricas (divisões por zero em pontos de inflexão) e validando ordens de grandeza em unidades de aceleração gravitacional ($g$).

---
### 🧠 Enquadramento Teórico: Coordenadas Intrínsecas

Numa trajetória arbitrária $\vec{r}(t)$, o vetor velocidade é sempre tangente ao caminho:
$$\vec{v}(t) = v(t)\,\hat{u}_t(t)$$
onde $v(t) = \|\vec{v}(t)\| = \frac{ds}{dt}$ é a rapidez (módulo da velocidade) e $\hat{u}_t = \frac{\vec{v}}{v}$ é o vetor unitário tangente.

Derivando no tempo pela regra do produto:
$$\vec{a}(t) = \frac{d\vec{v}}{dt} = \frac{dv}{dt}\,\hat{u}_t + v\,\frac{d\hat{u}_t}{dt}$$

Recorrendo à geometria diferencial plana, o vetor unitário tangente varia exclusivamente na direção perpendicular $\hat{u}_n$ dirigida para o centro de curvatura:
$$\frac{d\hat{u}_t}{dt} = \frac{v}{R}\,\hat{u}_n$$
onde $R$ é o **raio de curvatura** local (por vezes denotado $\rho$) e $\kappa = 1/R$ é a curvatura.

Assim, a aceleração decompõe-se ortogonalmente em:
$$\vec{a}(t) = a_t\,\hat{u}_t + a_n\,\hat{u}_n = \dot{v}\,\hat{u}_t + \frac{v^2}{R}\,\hat{u}_n$$

**Projeções Algébricas Práticas:**
- Aceleração tangencial: $a_t = \frac{\vec{a} \cdot \vec{v}}{v} = \frac{a_x v_x + a_y v_y}{\sqrt{v_x^2 + v_y^2}}$
- Aceleração normal: $a_n = \sqrt{\|\vec{a}\|^2 - a_t^2} = \frac{\|\vec{v} \times \vec{a}\|}{v} = \frac{v^2}{R}$
- Raio de curvatura: $R = \frac{v^2}{a_n} = \frac{\|\vec{v}\|^3}{\|\vec{v} \times \vec{a}\|}$
- Centro do círculo de curvatura: $\vec{C}(t) = \vec{r}(t) + R(t)\,\hat{u}_n(t)$

In [ ]:
# Bloco 1: Importação de Bibliotecas Científicas e Configuração Estética
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo visual dos gráficos (paleta institucional LEFA/ISEP)
plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Ambiente Python configurado com sucesso!')

---
### 📈 Secção 1: Trajetória Elíptica com Rapidez Variável

Vamos considerar uma partícula cuja posição descreve uma órbita elíptica:
$$\vec{r}(t) = (a \cos\omega t)\,\hat{\imath} + (b \sin\omega t)\,\hat{\jmath}$$

Diferenciando analiticamente:
$$\vec{v}(t) = (-a\omega \sin\omega t)\,\hat{\imath} + (b\omega \cos\omega t)\,\hat{\jmath}$$
$$\vec{a}(t) = (-a\omega^2 \cos\omega t)\,\hat{\imath} + (-b\omega^2 \sin\omega t)\,\hat{\jmath} = -\omega^2 \vec{r}(t)$$

Repara que a aceleração aponta sempre diretamente para a origem (força central elástica), mas **não coincide** com a normal da curva exceto nos 4 vértices da elipse!

In [ ]:
# Bloco 2: Modelação Vetorizada e Cálculo das Componentes Intrínsecas
def simular_elipse(a=10.0, b=5.0, omega=1.0, N_pontos=400):
    T = 2 * np.pi / omega
    t = np.linspace(0, T, N_pontos)
    
    # Posição
    x = a * np.cos(omega * t)
    y = b * np.sin(omega * t)
    
    # Velocidade cartesiana
    vx = -a * omega * np.sin(omega * t)
    vy = b * omega * np.cos(omega * t)
    v = np.sqrt(vx**2 + vy**2)
    
    # Aceleração cartesiana
    ax = -a * (omega**2) * np.cos(omega * t)
    ay = -b * (omega**2) * np.sin(omega * t)
    a_total = np.sqrt(ax**2 + ay**2)
    
    # Aceleração tangencial (projeção escalar: (a . v) / v)
    at = (ax * vx + ay * vy) / v
    
    # Aceleração normal: an = sqrt(a_total^2 - at^2)
    # Usamos np.maximum para evitar imprecisões numéricas ligeiramente negativas
    an = np.sqrt(np.maximum(0.0, a_total**2 - at**2))
    
    # Raio de curvatura: rho = v^2 / an
    rho = v**2 / an
    
    # Vetores unitários intrínsecos
    ut_x = vx / v
    ut_y = vy / v
    
    # un é obtido a partir de (a - at * ut) / an
    un_x = (ax - at * ut_x) / an
    un_y = (ay - at * ut_y) / an
    
    # Centro de curvatura C = r + rho * un
    cx = x + rho * un_x
    cy = y + rho * un_y
    
    return t, x, y, vx, vy, v, ax, ay, a_total, at, an, rho, ut_x, ut_y, un_x, un_y, cx, cy

# Execução da simulação para a = 10 m, b = 5 m, omega = 1 rad/s
t, x, y, vx, vy, v, ax, ay, a_total, at, an, rho, ut_x, ut_y, un_x, un_y, cx, cy = simular_elipse()

# Visualização estática dos perfis físicos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Trajetória no plano XY
axes[0].plot(x, y, color='#0f2b48', lw=2.5, label='Trajetória Elíptica')
axes[0].plot(0, 0, 'k+', ms=12, label='Centro O')
axes[0].scatter([10, -10, 0, 0], [0, 0, 5, -5], color='#d90429', zorder=5, label='Vértices da Curva')
axes[0].set_xlabel('$x$ [m]')
axes[0].set_ylabel('$y$ [m]')
axes[0].set_title('Trajetória Espacial $xy$')
axes[0].axis('equal')
axes[0].legend(loc='upper right', frameon=True)

# Subplot 2: Evolução temporal de a_t, a_n e rapidez v
axes[1].plot(t, at, color='#f77f00', lw=2, label='$a_t(t)$ [Tangencial]')
axes[1].plot(t, an, color='#0077b6', lw=2, label='$a_n(t)$ [Normal / Centrípeta]')
axes[1].plot(t, v, color='#2b9348', lw=1.8, ls='--', label='$v(t)$ [Rapidez]')
axes[1].axhline(0, color='gray', lw=0.8, ls=':')
axes[1].set_xlabel('Tempo $t$ [s]')
axes[1].set_ylabel('Módulo da Grandeza [m/s ou m/s²]')
axes[1].set_title('Componentes Intrínsecas da Aceleração e Rapidez')
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

print(f"Raio de curvatura no vértice maior (x=±a): rho = {rho[0]:.2f} m (Teórico b²/a = {5**2/10:.2f} m)")
print(f"Raio de curvatura no vértice menor (y=±b): rho = {rho[len(rho)//4]:.2f} m (Teórico a²/b = {10**2/5:.2f} m)")

---
### 🎥 Secção 2: Animação Dinâmica do Triedro Móvel e Círculo de Curvatura

A célula seguinte gera uma animação interativa com reprodutor de vídeo HTML5 diretamente no Colab.

**Legenda dos Vetores:**
- 🟢 **Vetor Verde:** Velocidade $\vec{v}$ (sempre tangente à curva, direção $\hat{u}_t$);
- 🔵 **Vetor Azul Escuro:** Aceleração total $\vec{a}$ (aponta sempre para o interior da curva);
- 🟠 **Vetor Laranja:** Componente tangencial $a_t \hat{u}_t$ (faz o carro acelerar ou travar);
- 🔷 **Vetor Ciano:** Componente normal $a_n \hat{u}_n$ (faz o carro mudar de direção);
- 🟣 **Círculo Tracejado Magenta:** **Círculo de Curvatura** local, de raio $R(t)$ centrado em $\vec{C}(t)$.

In [ ]:
# Bloco 3: Construção e Renderização da Animação Interativa do Triedro de Frenet-Serret
fig_anim, ax_anim = plt.subplots(figsize=(8, 7))

# Trajetória de fundo
ax_anim.plot(x, y, color='#cbd5e1', lw=3, label='Trajetória')
ponto_particula, = ax_anim.plot([], [], 'o', color='#0f2b48', ms=9, zorder=6)
circulo_curvatura = plt.Circle((0, 0), 1.0, color='#9d4edd', fill=False, ls='--', lw=2, zorder=4, label='Círculo de Curvatura')
ax_anim.add_patch(circulo_curvatura)
ponto_centro, = ax_anim.plot([], [], 'x', color='#9d4edd', ms=8, mew=2, zorder=5, label='Centro Curvatura C')

# Linhas para os vetores (usamos quiver ou lines para velocidade e acelerações)
escala_v = 0.4
escala_a = 0.5

line_v, = ax_anim.plot([], [], color='#2b9348', lw=3, label='Velocidade v (Tangente)')
line_a, = ax_anim.plot([], [], color='#0f2b48', lw=2.5, ls='-', label='Aceleracao Total a')
line_at, = ax_anim.plot([], [], color='#f77f00', lw=2, ls=':', label='Comp. Tangencial a_t')
line_an, = ax_anim.plot([], [], color='#00b4d8', lw=2, ls=':', label='Comp. Normal a_n')

ax_anim.set_xlim(-22, 22)
ax_anim.set_ylim(-22, 22)
ax_anim.set_aspect('equal')
ax_anim.set_xlabel('$x$ [m]')
ax_anim.set_ylabel('$y$ [m]')
ax_anim.set_title('Triedro Movel (u_t, u_n) e Circulo de Curvatura Dinamico')
ax_anim.legend(loc='upper right', fontsize=9, frameon=True)

# Subamostragem para a animação ter 60 frames suaves
indices = np.linspace(0, len(t) - 1, 60, dtype=int)

def init():
    ponto_particula.set_data([], [])
    ponto_centro.set_data([], [])
    circulo_curvatura.set_visible(False)
    line_v.set_data([], [])
    line_a.set_data([], [])
    line_at.set_data([], [])
    line_an.set_data([], [])
    return ponto_particula, ponto_centro, circulo_curvatura, line_v, line_a, line_at, line_an

def animate(i):
    idx = indices[i]
    px, py = x[idx], y[idx]
    
    # Posição da partícula
    ponto_particula.set_data([px], [py])
    
    # Círculo de Curvatura
    rad = rho[idx]
    ccx, ccy = cx[idx], cy[idx]
    circulo_curvatura.center = (ccx, ccy)
    circulo_curvatura.set_radius(rad)
    circulo_curvatura.set_visible(True)
    ponto_centro.set_data([ccx], [ccy])
    
    # Vetores
    line_v.set_data([px, px + vx[idx] * escala_v], [py, py + vy[idx] * escala_v])
    line_a.set_data([px, px + ax[idx] * escala_a], [py, py + ay[idx] * escala_a])
    
    # Componentes at e an
    line_at.set_data([px, px + at[idx] * ut_x[idx] * escala_a], [py, py + at[idx] * ut_y[idx] * escala_a])
    line_an.set_data([px, px + an[idx] * un_x[idx] * escala_a], [py, py + an[idx] * un_y[idx] * escala_a])
    
    return ponto_particula, ponto_centro, circulo_curvatura, line_v, line_a, line_at, line_an

anim = FuncAnimation(fig_anim, animate, init_func=init, frames=len(indices), interval=60, blit=True)
plt.close(fig_anim)  # Fecha a figura estática

# Exibe como animação JavaScript interativa no Colab
HTML(anim.to_jshtml())

---
### 🏎️ Secção 3: Desafio GenAI — Modelação de Chicane de F1 e Auditoria Física

#### 📝 Enunciado do Desafio:
Um monolugar de Fórmula 1 atravessa a chicane de Monza (Variante del Rettifilo). O traçado espacial é aproximado pela função sinusoidal:
$$y(x) = H \sin\left(\frac{2\pi x}{L}\right) \quad \text{com } x \in [0, L]$$
onde $H = 6.0\text{ m}$ e $L = 60.0\text{ m}$.

O piloto entra a $v_0 = 45\text{ m/s}$ ($162\text{ km/h}$) e trava linearmente até sair a $v_f = 30\text{ m/s}$ ($108\text{ km/h}$):
$$v(x) = v_0 - \Delta v \left(\frac{x}{L}\right)$$

#### 🤖 Engenharia de Prompt para IA (Claude / ChatGPT / Gemini):
Copia o seguinte prompt e submete a um modelo de IA:

> *"Escreve uma função Python vetorizada usando NumPy e Matplotlib para modelar a cinemática de um carro de F1 numa curva sinusoidal y(x) = H * sin(2*pi*x / L). A função deve:*  
> *(1) Calcular a primeira e segunda derivadas analíticas y'(x) e y''(x);*  
> *(2) Calcular o raio de curvatura em cada ponto: rho(x) = (1 + y'(x)**2)**(1.5) / abs(y''(x));*  
> *(3) Calcular a aceleração normal a_n(x) = v(x)**2 / rho(x) em m/s^2 e em 'g' (dividindo por 9.8);*  
> *(4) Traçar dois gráficos: o traçado da pista colorido pela aceleração lateral e a evolução de a_n(x).*  
> *Trata singularidades onde y'' = 0 substituindo rho por infinito."*

#### 🔍 Checklist de Auditoria Física:
1. **Pontos de Inflexão ($x = 0, L/2, L$):** $y''(x) = 0 \implies \rho \to \infty \implies a_n = 0$. O código da IA lidou com a divisão por zero sem quebrar (`ZeroDivisionError`)?
2. **Picos de Aceleração Lateral:** Ocorrem perto dos ápices da curva ($x \approx L/4$ e $x \approx 3L/4$), onde a curvatura é máxima.
3. **Ordem de Grandeza em $g$:** A aceleração lateral máxima deve situar-se entre $3.5\,g$ e $4.5\,g$, perfeitamente dentro dos limites dos pneus Pirelli de F1 com efeito solo!

In [ ]:
# Bloco 4: Implementação de Referência e Auditoria da Chicane de F1
def simular_chicane_f1(H=6.0, L=60.0, v_in=45.0, v_out=30.0, N=500):
    x = np.linspace(0, L, N)
    k = 2 * np.pi / L
    
    # Geometria da chicane e derivadas analíticas
    y = H * np.sin(k * x)
    y_prime = H * k * np.cos(k * x)          # dy/dx
    y_double_prime = -H * (k**2) * np.sin(k * x) # d²y/dx²
    
    # Perfil de velocidade com travagem
    v = v_in - (v_in - v_out) * (x / L)
    
    # Cálculo do raio de curvatura rho = (1 + y'^2)^(3/2) / |y''|
    # Proteção de singularidade (onde y'' = 0 nos pontos de inflexão)
    tolerancia = 1e-5
    denominador = np.abs(y_double_prime)
    
    rho = np.empty_like(x)
    singular = denominador < tolerancia
    rho[singular] = np.inf
    rho[~singular] = ((1 + y_prime[~singular]**2)**1.5) / denominador[~singular]
    
    # Aceleração normal: an = v^2 / rho (em m/s² e em unidades de g)
    an = np.zeros_like(x)
    an[~singular] = (v[~singular]**2) / rho[~singular]
    an_g = an / 9.80
    
    return x, y, v, rho, an, an_g

# Execução da simulação da chicane
x_f1, y_f1, v_f1, rho_f1, an_f1, an_g_f1 = simular_chicane_f1()

# Visualização dos resultados com mapa de cores
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Gráfico 1: Pista com código de cores da aceleração em g
scatter = ax1.scatter(x_f1, y_f1, c=an_g_f1, cmap='plasma', s=25, edgecolor='none')
cbar = plt.colorbar(scatter, ax=ax1, pad=0.02)
cbar.set_label('Aceleração Lateral [$g$]', rotation=90)
ax1.set_ylabel('$y$ [m]')
ax1.set_title('Traçado da Chicane com Gradiente de Carga Lateral ($g$)')
ax1.axis('equal')

# Gráfico 2: Evolução de an (em g) ao longo da chicane
ax2.plot(x_f1, an_g_f1, color='#d90429', lw=2.2, label='Aceleração Normal $a_n$')
ax2.axhline(5.0, color='black', ls='--', lw=1.2, label='Limite Fisico Pneu F1 (~5g)')
ax2.set_xlabel('Distância Longitudinal $x$ [m]')
ax2.set_ylabel('Aceleração Lateral [$g$]')
ax2.set_title('Perfil de Aceleração Centrípeta ao Longo da Chicane')
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

# Relatório de auditoria física
idx_max = np.argmax(an_g_f1)
print(f"🏁 RESULTADO DA AUDITORIA FÍSICA F1:")
print(f"   - Aceleração lateral máxima: {an_g_f1[idx_max]:.2f} g ({an_f1[idx_max]:.1f} m/s²)")
print(f"   - Localização do pico: x = {x_f1[idx_max]:.1f} m (1.ª perna da chicane)")
print(f"   - Raio mínimo de curvatura: rho_min = {np.min(rho_f1):.2f} m")
print(f"   - Rapidez no ponto crítico: v = {v_f1[idx_max]:.1f} m/s ({v_f1[idx_max]*3.6:.1f} km/h)")
print(f"   - Veredito Físico: Perfeitamente viável para F1 atual com aerodinâmica de solo (downforce)!")

---
### 📝 Questões de Reflexão para a Folha de Turno:
1. **Comparação de Picos:** Porque é que o pico de aceleração lateral na 1.ª perna da chic ($x \approx 15\text{ m}$) é superior ao da 2.ª perna ($x \approx 45\text{ m}$), apesar da curvatura geométrica $|y''|$ ser idêntica nos dois ápices?
2. **Efeito da Velocidade:** Se o piloto tentasse fazer a chicane a $v = 60\text{ m/s}$ ($216\text{ km/h}$) sem travar, qual seria a aceleração lateral máxima? O carro conseguiria curvar?